<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 1.5 · Más variables y la temporada

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Tener en cuenta la **temporada** con un índice del mes.
- Dar **varias variables** a un modelo.
- Convertir **textos en números**.
- Comparar el resultado por cliente y por cartera de vendedor.

## 📊 En Excel lo harías así

Sabes que diciembre vende más que febrero. Entonces:

1. Armas una **tabla dinámica** con las ventas de cada mes de 2024.
2. Divides cada mes entre el promedio mensual: ese es su **índice**. Si diciembre da 1.47, vende 47 % más que un mes normal.
3. Multiplicas tu proyección por el índice del mes.

Veamos si con eso basta, o si un modelo con más datos lo hace mejor.

## 🐍 En Python

### Paso 1: preparar

La misma tabla del episodio 1.4, con la misma separación por fechas. `.copy()` crea una tabla aparte, para poder agregarle columnas sin que pandas se queje.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

entreno = tabla[tabla["mes"] < "2025-07"]
prueba = tabla[tabla["mes"] >= "2025-07"].copy()

print(len(entreno), len(prueba))

3475 1793


### Paso 2: la regla con índice

El índice sale de 2024, así que no usa nada del periodo de prueba. `.str[5:7]` toma el mes de la fecha:

In [4]:
ventas_2024 = ventas[ventas["fecha"].str[:4] == "2024"]
por_mes = ventas_2024.groupby(ventas_2024["fecha"].str[5:7].astype(int))["total"].sum()

indice = por_mes / por_mes.mean()

indice.round(2)

fecha
1     0.73
2     0.80
3     0.82
4     0.99
5     1.00
6     0.97
7     1.27
8     1.02
9     0.87
10    1.00
11    1.07
12    1.47
Name: total, dtype: float64

Julio y diciembre destacan. Ahora multiplicamos el promedio de 6 meses por el índice del mes. `.map()` busca en `indice` el valor de cada mes, como un `=BUSCARV()` (`VLOOKUP`):

In [5]:
prueba["regla_indice"] = prueba["promedio_6"] * prueba["numero_mes"].map(indice)

print(round(mean_absolute_error(prueba["total"], prueba["promedio_6"])))
print(round(mean_absolute_error(prueba["total"], prueba["regla_indice"])))

855
827


Bajó de S/ 855 a S/ 827, con una fórmula de Excel.

### Paso 3: textos en números

Un modelo solo entiende números, pero `tipo` y `ciudad` son textos. La solución es una columna por cada valor, con 1 si la fila lo tiene y 0 si no. `numero_mes` también se trata así: diciembre no es "12 veces enero", es otra categoría.

Esta función lista lo hace con `pd.get_dummies()`. Mira el resultado:

In [6]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [7]:
numeros = ["mes_anterior", "promedio_3"]
textos = ["tipo", "ciudad", "numero_mes"]

x_entreno = preparar_x(entreno, numeros, textos)

print(x_entreno.shape)
list(x_entreno.columns[:6])

(3475, 23)


['mes_anterior',
 'promedio_3',
 'tipo_Bodega',
 'tipo_Minimarket',
 'tipo_Restaurante',
 'ciudad_Arequipa']

> ⚠️ En la prueba hay que usar **las mismas columnas** que en entrenamiento. Aquí, por ejemplo, no hay enero: sin cuidado, faltaría esa columna y el modelo daría un error. Por eso le pasamos `x_entreno.columns`.

### Paso 4: el modelo con varias variables

Es igual que antes. `.fit()` ahora recibe 23 columnas, en vez de una:

In [8]:
x_prueba = preparar_x(prueba, numeros, textos, x_entreno.columns)

modelo = LinearRegression()
modelo.fit(x_entreno, entreno["total"])

prueba["regresion"] = modelo.predict(x_prueba)

print(round(mean_absolute_error(prueba["total"], prueba["regresion"])))

921


S/ 921. ¡Peor que la regla con índice! Cliente por cliente, **una regla simple le gana a esta regresión**.

### Paso 5: ¿y por cartera?

A un gerente le importa más el total de cada vendedor, en cada mes, que cada cliente por separado. Al sumar, los errores de distintos clientes se compensan en parte. Sumamos con `groupby`, como en el episodio 0.6, pero con una lista de dos columnas:

In [9]:
cartera = prueba.groupby(["id_vendedor", "mes"])[
    ["total", "promedio_6", "regla_indice", "regresion"]
].sum()

print(round(cartera["total"].mean()))
print(round(mean_absolute_error(cartera["total"], cartera["promedio_6"])))
print(round(mean_absolute_error(cartera["total"], cartera["regla_indice"])))
print(round(mean_absolute_error(cartera["total"], cartera["regresion"])))

45043
7704
4963
4819


Una cartera vende unos S/ 45 mil al mes. Sin temporada, el error es de unos S/ 7,700. Con temporada baja a cerca de S/ 4,900, **ya sea con el modelo o con la regla**: S/ 4,819 y S/ 4,963 son prácticamente un empate.

¿Y qué aporta cada variable al modelo? Probando con y sin ellas:

| Además de `mes_anterior` y `promedio_3` | Error por cliente | Error por cartera |
|---|---|---|
| Nada | 921 | 8,015 |
| Mes | 916 | 4,781 |
| Tipo y ciudad | 930 | 7,974 |
| Los tres | 921 | 4,819 |

Conclusiones honestas:

- **Lo que más ayuda es tener en cuenta la temporada**, y solo por cartera. Cerca del 80 % de esa mejora viene de julio y diciembre.
- **Más variables no garantizan un mejor resultado.** El tipo y la ciudad no aportan aquí. Compara siempre.
- **Es una sola muestra:** solo hay un julio y un diciembre en la prueba. No lo tomes como una ley.

## 🤖 Pídeselo a la IA

> *"Tengo una columna de texto `region` con 4 valores y quiero usarla en una regresión lineal de scikit-learn. Explícame sin fórmulas qué hace `pd.get_dummies` y qué problema aparece si en los datos nuevos falta una región."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Índice del mes con una tabla dinámica | `por_mes / por_mes.mean()` |
| `=BUSCARV()` (`VLOOKUP`) del índice | `.map(indice)` |
| Una columna por cada valor de un texto | `pd.get_dummies()` |
| Sumar por vendedor y mes | `groupby(["id_vendedor", "mes"])` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/ejercicio.ipynb) e intenta mejorar el pronóstico. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0